# 🔑 Notebook 2: Idempotency Keys

The client generates a unique key (a UUID) for each *logical* request and sends it in a header like `Idempotency-Key: <uuid>`. The server stores `key -> result` the first time. Subsequent retries with the same key get the **cached** result back — no side effect.

This is how Stripe, AWS, GitHub, Shopify, PayPal, and most modern APIs handle retries safely.

## 🧭 Mental model

```
  client                                server
  ------                                ------
  POST /charge                          
  Idempotency-Key: 9f3a…                
  { amount: 10 }          ───────►      first time? apply, cache, reply
                          ◄───────      200 { balance: 90 }   (response lost 😵)

  POST /charge (retry)                  
  Idempotency-Key: 9f3a…                
  { amount: 10 }          ───────►      seen key → return cached reply
                          ◄───────      200 { balance: 90 }   ← exactly-once effect
```


## 🛠️ Setup

```bash
cd 04-patterns/idempotency
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 🟩 GOOD: in-memory replay cache

In [1]:
import uuid

class PaymentService:
    def __init__(self):
        self.balances = {'alice': 100}
        self.idem = {}  # key -> stored result

    def charge(self, key, account, amount):
        if key in self.idem:
            print(f'  ↩ replay for {key[:8]}… — returning cached response')
            return self.idem[key]
        self.balances[account] -= amount
        result = {'ok': True, 'balance': self.balances[account], 'charged': amount}
        self.idem[key] = result
        return result

svc = PaymentService()
k = str(uuid.uuid4())
print('1st:', svc.charge(k, 'alice', 10))
print('2nd:', svc.charge(k, 'alice', 10))
print('3rd:', svc.charge(k, 'alice', 10))
print(f"\nAlice ended with {svc.balances['alice']} — charged exactly once ✅")


1st: {'ok': True, 'balance': 90, 'charged': 10}
  ↩ replay for a4b554d4… — returning cached response
2nd: {'ok': True, 'balance': 90, 'charged': 10}
  ↩ replay for a4b554d4… — returning cached response
3rd: {'ok': True, 'balance': 90, 'charged': 10}

Alice ended with 90 — charged exactly once ✅


A *different* key charges again — because that's a *different logical request*:

In [2]:
k2 = str(uuid.uuid4())  # brand-new key, brand-new intent
print('new key:', svc.charge(k2, 'alice', 5))
print(f"Alice now at {svc.balances['alice']} — second *logical* charge applied.")


new key: {'ok': True, 'balance': 85, 'charged': 5}
Alice now at 85 — second *logical* charge applied.


## 🟥 Gotcha: same key, *different* body

What if a buggy client reuses the same key but with a different amount? A naive cache would happily return the first result — silently ignoring the new request. Stripe rejects this with a `400`. We should too.

The fix: store a **fingerprint** (hash) of the request body alongside the key, and compare on replay.

In [3]:
import hashlib, json

class SaferPaymentService:
    def __init__(self):
        self.balances = {'alice': 100}
        self.idem = {}  # key -> (body_hash, result)

    @staticmethod
    def _fingerprint(body: dict) -> str:
        # sort_keys so {'a':1,'b':2} and {'b':2,'a':1} hash the same
        return hashlib.sha256(json.dumps(body, sort_keys=True).encode()).hexdigest()

    def charge(self, key, body):
        fp = self._fingerprint(body)
        if key in self.idem:
            cached_fp, cached_result = self.idem[key]
            if cached_fp != fp:
                raise ValueError(
                    '409 Conflict: idempotency key reused with a different body'
                )
            return cached_result
        self.balances[body['account']] -= body['amount']
        result = {'ok': True, 'balance': self.balances[body['account']]}
        self.idem[key] = (fp, result)
        return result

svc = SaferPaymentService()
k = str(uuid.uuid4())
print('first  $10:', svc.charge(k, {'account': 'alice', 'amount': 10}))
print('replay $10:', svc.charge(k, {'account': 'alice', 'amount': 10}))
try:
    svc.charge(k, {'account': 'alice', 'amount': 999})  # SAME key, DIFFERENT body
except ValueError as e:
    print('rejected:', e)


first  $10: {'ok': True, 'balance': 90}
replay $10: {'ok': True, 'balance': 90}
rejected: 409 Conflict: idempotency key reused with a different body


## 🧹 Operational notes

- Put the key in a header (`Idempotency-Key`) so middleware can enforce it uniformly — applications shouldn't have to remember to check.
- The replay cache grows forever; expire keys after the **retry window** (Stripe uses 24h).
- In-memory only works for a single process. Multiple replicas + restarts = 👉 notebook 3 (durable, transactional dedup).
